This notebook covers the following topics:
1. Adding a wrapper for your model to [fev/models](https://github.com/autogluon/fev/tree/main/models).
2. Submitting the results for your model to the [fev-leaderboard](https://huggingface.co/spaces/autogluon/fev-leaderboard).

## Adding a wrapper for your model

Each model wrapper lives in its own subfolder under `models/`. The evaluation harness (`models/evaluate.py`) discovers and runs them automatically.

### Step 1: Create the folder

Create a folder `models/<name>/` where `<name>` is how you'll refer to the model with the `-m` flag.

### Step 2: Add `model.py`

Implement a subclass of `fev.ForecastingModel`. The `model_name` class attribute must match the folder name.

```python
# models/my-model/model.py
import datasets

import fev


class MyModel(fev.ForecastingModel):
    model_name = "my-model"  # must match the folder name

    # List HF dataset configs (from autogluon/fev_datasets) used during pretraining.
    # Used to flag potential data leakage. Leave empty for models that train from scratch.
    trained_on_datasets = ["kdd_cup_2022_10T", "m5_1D"]

    def __init__(self, model_size: str = "small"):
        super().__init__()
        self.model_size = model_size

    def _fit_predict(self, task: fev.Task) -> list[datasets.DatasetDict]:
        predictions_per_window = []
        for window in task.iter_windows():
            past_data, future_data = window.get_input_data()

            with self._record_inference_time():
                # Generate predictions for each time series
                predictions = {"predictions": [...]}

            predictions_per_window.append(predictions)
        return predictions_per_window
```

Key points about `_fit_predict`:
- Called once per task. Must return predictions for **all** evaluation windows.
- Use `self._record_inference_time()` context manager to track inference time.
- Use `self._record_training_time()` if your model has a training step.
- Each call should be independent — don't carry over state from prior tasks.
- Caching expensive resources (weights, tokenizers) on `self` across calls is fine.

### Step 3: Add `requirements.txt`

List pinned dependencies for your model. These are installed automatically in an ephemeral environment when running `evaluate.py` — your project environment is not modified.

```
# models/my-model/requirements.txt
my-forecasting-lib==1.2.3
torch>=2.0
```

### Predictions format

Predictions must follow the schema provided by `task.predictions_schema`.

In [ ]:
import fev

task = fev.Task(
    dataset_path="autogluon/chronos_datasets",
    dataset_config="monash_rideshare",
    target="price_mean",
    horizon=30,
)
task.predictions_schema

For probabilistic forecasting tasks (when `task.quantile_levels` is set), predictions must additionally contain quantile forecasts:

In [ ]:
task = fev.Task(
    dataset_path="autogluon/chronos_datasets",
    dataset_config="monash_rideshare",
    target="price_mean",
    horizon=30,
    quantile_levels=[0.1, 0.5, 0.9],
)
task.predictions_schema

Predictions cannot contain any `NaN` values.

### Tips

- If your model generates probabilistic forecasts, choose the "optimal" point forecast for the `task.eval_metric`. For example, metrics like `"MSE"` prefer the mean, while `"MASE"` is optimized by the median.
- Use `fev.convert_input_data()` to take advantage of adapters and reduce boilerplate preprocessing.
- Make sure your wrapper handles missing values (or imputes them before passing data to the model).
- Take advantage of extra features available via `task.static_columns`, `task.dynamic_columns`, `task.known_dynamic_columns`, and `task.past_dynamic_columns`.

## Running evaluation

```bash
python models/evaluate.py -m my-model
```

Options:
- `-m` — model name (must match a subfolder in `models/`)
- `-b` — path or URL to benchmark YAML (default: `fev_bench_mini`)
- `-n` — display name for results (default: same as `-m`)
- `-k` — JSON dict of kwargs passed to the model constructor
- `-t` — limit number of tasks (useful for quick testing)

## Submitting results to the leaderboard

After implementing your model wrapper, follow these steps to submit results to the [fev-leaderboard](https://huggingface.co/spaces/autogluon/fev-leaderboard):

1. Fork [`autogluon/fev`](https://github.com/autogluon/fev) and clone your fork.
2. Implement your model wrapper in `models/<name>/`.
3. Run the model on all tasks from the benchmark and save results:
   ```bash
   python models/evaluate.py -m <name> -b benchmarks/fev_bench/tasks.yaml
   mv <name>.csv benchmarks/fev_bench/results/<name>.csv
   ```
4. Open a pull request to `autogluon/fev` containing:
   - `models/<name>/model.py`
   - `models/<name>/requirements.txt`
   - `benchmarks/fev_bench/results/<name>.csv`
5. We will independently reproduce the results using your code and add them to the leaderboard.